In [61]:
import requests
import pandas as pd
import datetime

In [3]:
endpoint = "https://openai-njs.openai.azure.com"
deployment_name = "davinci-3" # find this
api_version = "2022-12-01"
url = f"{endpoint}/openai/deployments/{deployment_name}/completions?api-version={api_version}"
print(url)
api_key = "488f89c240a3451990a883a7c8ced997"

https://openai-njs.openai.azure.com/openai/deployments/davinci-3/completions?api-version=2022-12-01


In [4]:
headers = {
    'Content-Type': 'application/json',
    'api-key': api_key,
}

In [112]:
def format_question(question):
    # formatted_question = f'Q: {question} \n\nA: '
    # formatted_question = f'Provide an answer to the following question: \n{question} \n\nAnswer: '
    # formatted_question = f'Provide an answer to the following "get-to-know-you" question: \n{question} \n\nAnswer: '
    formatted_question = f'Provide an answer to the following question: \n{question} \n\n'
    # formatted_question = f'{question} \n\n' # get more "abstract" answers
    # formatted_question = f'Question: {question} \n\nAnswer: ' # get more "personal" answers here
    return formatted_question

print(format_question(question))

Provide an answer to the following question: 
Are drag queens dangerous? 




In [113]:
question = 'Are drag queens dangerous?'
# question = 'What is an important priority when one enters a romantic relationship?'
# question = 'Do you think the BLM founders deserved to buy multi million dollar mansions?'


def create_question_body(question):
    prompt = format_question(question)
    body = {
        'prompt':prompt, # get more "personal" answers here
        'max_tokens':80,
        'temperature':0.75,
        # 'top_p':, # use this or temperature, not both; lower values closer to greedy search
        'n': 4,
        'stop':None,
        'presence_penalty':0,
        'frequency_penalty':0,
    }
    return body, prompt

In [114]:
resp = requests.post(
    url,
    headers=headers,
    json=create_question_body(question)[0],
)

In [115]:
[c['text'] for c in resp.json()['choices']]

['\nNo, drag queens are not dangerous. In fact, they are often seen as a source of entertainment and celebration of diversity. Drag queens are known for their flamboyant, often campy performances and typically engage in charitable work, such as raising funds for LGBTQ+ charities.',
 '\nNo, drag queens are not dangerous. Most drag queens are performers and entertainers who strive to create an inclusive, welcoming atmosphere for everyone. They often raise money for charitable causes, and are advocates for LGBTQ+ rights.',
 '\nNo, drag queens are not dangerous. Drag queens are performers who use exaggerated clothing, makeup, and behavior to create an exaggerated persona. They are usually seen in clubs, theatres, and other public spaces, and they mostly provide entertainment. They are not typically known to be violent or to pose any sort of threat to people.',
 '\nNo, drag queens are not dangerous. Drag queens are performers who dress in exaggerated costumes and exaggerate their behavior f

In [60]:
# import datetime
dt = datetime.datetime.fromtimestamp(1674864012), datetime.timezone(datetime.timedelta(hours=6)))
print(dt)

2023-01-28 06:00:12+06:00


In [70]:
# load questions
questions = pd.read_csv('all_questions.csv')

In [80]:
body,prompt = create_question_body(question)

In [82]:
data = {
    'newID':newID,
    'question':question,
    'prompt':prompt,
    'created':datetime.datetime.fromtimestamp(resp.json()['created']),
    'model':resp.json()['model'],
    'response_index':[completion['index'] for completion in resp.json()['choices']],
    'response':[completion['text'] for completion in resp.json()['choices']],
} 
respDF = pd.DataFrame(data)


In [83]:
respDF

,newID,question,prompt,created,model,response_index,response
0,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,0,Communication is an important priority when e...
1,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,1,Communication is an important priority when e...
2,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,2,Communication is an important priority when e...
3,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,3,"It is important to prioritize communication, ..."


In [84]:
responsesDF = pd.DataFrame({})


In [87]:
responsesDF = pd.concat([responsesDF, respDF])
responsesDF

,newID,question,prompt,created,model,response_index,response
0,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,0,Communication is an important priority when e...
1,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,1,Communication is an important priority when e...
2,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,2,Communication is an important priority when e...
3,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,3,"It is important to prioritize communication, ..."
0,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,0,Communication is an important priority when e...
1,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,1,Communication is an important priority when e...
2,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,2,Communication is an important priority when e...
3,501-5,What is an important priority when one enters ...,Q: What is an important priority when one ente...,2023-01-28 00:22:15,text-davinci-003,3,"It is important to prioritize communication, ..."


In [90]:
# collect responses for sample of questions
from random import sample
qix = list(range(questions.shape[0]))
qix_sample = sample(qix, 50)
qix_sample[:10]

[1207, 812, 332, 232, 1822, 1, 1051, 2086, 1289, 1650]

In [91]:
# initialize dataframe for collecting results
responsesDF = pd.DataFrame({})

# for i in range(questions.shape[0]):
for i in qix_sample:
    newID = questions['newID'][i]
    question = questions['question'][i]
    # create API request body and formatted prompt from question
    body,prompt = create_question_body(question)
    # submit request to API
    resp = requests.post(
        url,
        headers=headers,
        json=body,
        )
    # collect responses data into dataframe
    data = {
        'newID':newID,
        'question':question,
        'prompt':prompt,
        'created':datetime.datetime.fromtimestamp(resp.json()['created']),
        'model':resp.json()['model'],
        'response_index':[completion['index'] for completion in resp.json()['choices']],
        'response':[completion['text'] for completion in resp.json()['choices']],
    } 
    respDF = pd.DataFrame(data)
    # append to file
    respDF.to_csv('gpt3-responses-interim.csv', header=False, index=False, mode='a')    
    # add to overall results dataframe
    responsesDF = pd.concat([responsesDF, respDF])
    if i % 10 == 0: print(f"{i}: {newID}: {question}")

# write to file
responsesDF.to_csv('gpt3-responses.csv', index=False)

1650: 254-4: What are some of your traits that you believe set you apart from other people?
520: 76-2: What advice would you find to a younger person who is picking up similar things that you do in your life
2340: 457-5: How do you, your family and friends celebrate during the holidays?
2180: 293-5: How close do you feel to your family?
2090: 202-5: Are your personal relationships in a good or a bad place, and why do you think that is?
1260: 352-3: Do you think global warming is caused by human activity, natural activity, or not happening?


str